# Training- sentences relating to eligibility

MiniLM embeddings + one logistic regression per label
(location, business_type, audience). MiniLM itself is not changed; the
classifiers learn what each kind of requirement looks like in its vector
space.

Folder layout (assuming work in colab/jupyter):
    grant_utils.py
    grants/   grant .txt files; the file name (without .txt) is the grant_id
    labels/   sentences.csv (your labels), to_label.csv (made by Cell 3)
    models/   first_pass.joblib (made by Cell 7)

Workflow:
  1. Run Cells 1-3 to make labels/to_label.csv, pre-filled by zero-shot.
  2. Download the csv thats created and hand-correct. uses binary 0 and 1 for relevance to related category (e.g. location, business)
  3. rename and add file under labels folder, named "sentences.csv" In Excel/Sheets Save As "CSV"
  3. Run Cells 4-7 to train, compare against zero-shot to see when good enough to save

  -> will output trained classifiers that can be layered on top of the MiniLM (or probably other) model

In [ ]:
# One-time setup: run this, then Kernel -> Restart. Skip it after that.
%pip install --user sentence-transformers scikit-learn pandas

In [ ]:
# ---------------------------------------------------------------------------
# CELL 1: imports and settings
# ---------------------------------------------------------------------------
import random
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score
from sklearn.model_selection import GroupKFold

from grant_utils import (
    EMBED_MODEL, LABEL_COLUMNS, QUERIES,
    embed, read_grant, split_sentences, zero_shot_scores,
)

GRANTS_DIR = Path("grants")
LABEL_CSV = Path("labels/sentences.csv")
TO_LABEL_CSV = Path("labels/to_label.csv")
MODEL_OUT = Path("models/first_pass.joblib")

# Held-out test grants (grant_ids). These are never trained on.
# Leave empty and Cell 4 picks ~20% at random and prints them: paste that
# list here so the test set stays the same every run.
TEST_GRANTS = []

TARGET_RECALL = 0.90      # threshold is set so ~90% of requirements get caught
ZERO_SHOT_MIN_SIM = 0.30  # same cutoff the zero-shot first pass uses
MIN_POSITIVES = 5         # labels with fewer training examples stay zero-shot

In [ ]:
# ---------------------------------------------------------------------------
# CELL 2: load the embedding model
# ---------------------------------------------------------------------------
embedder = SentenceTransformer(EMBED_MODEL)

In [ ]:
# ---------------------------------------------------------------------------
# CELL 3: make a labeling sheet for grants that aren't labeled yet.
# Pre-filled with zero-shot guesses (1 = looks like that kind of requirement).
# ---------------------------------------------------------------------------
already_labeled = set()
if LABEL_CSV.exists():
    already_labeled = set(
        pd.read_csv(LABEL_CSV, encoding="utf-8-sig")["grant_id"].astype(str)
    )

rows = []
for path in sorted(GRANTS_DIR.glob("*.txt")):
    grant_id = path.stem
    if grant_id in already_labeled:
        continue
    sentences = split_sentences(read_grant(path))
    if not sentences:
        continue
    vecs = embed(embedder, sentences)
    guesses = {
        col: zero_shot_scores(embedder, vecs, QUERIES[col]) >= ZERO_SHOT_MIN_SIM
        for col in LABEL_COLUMNS
    }
    for i, sentence in enumerate(sentences):
        rows.append({
            "grant_id": grant_id,
            "sentence": sentence,
            **{col: int(guesses[col][i]) for col in LABEL_COLUMNS},
        })

if rows:
    TO_LABEL_CSV.parent.mkdir(parents=True, exist_ok=True)
    sheet = pd.DataFrame(rows)
    sheet.to_csv(TO_LABEL_CSV, index=False, encoding="utf-8-sig")
    print(f"Wrote {len(sheet)} sentences from {sheet.grant_id.nunique()} "
          f"grants to {TO_LABEL_CSV}")
else:
    print("Every grant in grants/ is already labeled.")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 4: load the labels and split train / test BY GRANT
# ---------------------------------------------------------------------------
if not LABEL_CSV.exists():
    raise FileNotFoundError(
        f"No labels yet. Correct {TO_LABEL_CSV}, then save it as {LABEL_CSV}."
    )
df = pd.read_csv(LABEL_CSV, encoding="utf-8-sig")
missing = {"grant_id", "sentence", *LABEL_COLUMNS} - set(df.columns)
if missing:
    raise ValueError(f"{LABEL_CSV} is missing columns: {missing}")

df = df.dropna(subset=["sentence"]).reset_index(drop=True)
df["grant_id"] = df["grant_id"].astype(str)
df[LABEL_COLUMNS] = df[LABEL_COLUMNS].fillna(0).astype(int)

all_grants = sorted(df.grant_id.unique())
if not TEST_GRANTS:
    rng = random.Random(42)
    TEST_GRANTS = sorted(rng.sample(all_grants, max(1, round(0.2 * len(all_grants)))))
    print("No TEST_GRANTS set. Picked:", TEST_GRANTS)
    print("Paste that list into Cell 1 so it stays fixed.\n")
unknown = set(TEST_GRANTS) - set(all_grants)
if unknown:
    print("Warning: these TEST_GRANTS have no labeled rows:", sorted(unknown))

vecs = embed(embedder, df.sentence)
is_test = df.grant_id.isin(TEST_GRANTS).to_numpy()
train_df, test_df = df[~is_test], df[is_test]
X_train, X_test = vecs[~is_test], vecs[is_test]
train_groups = train_df.grant_id.to_numpy()

print(f"Train: {train_df.grant_id.nunique()} grants, {len(train_df)} sentences")
print(f"Test:  {test_df.grant_id.nunique()} grants, {len(test_df)} sentences")
print("Positive sentences (train / test):")
for col in LABEL_COLUMNS:
    print(f"  {col:<14} {train_df[col].sum():>4} / {test_df[col].sum()}")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 5: train one classifier per label.
# The threshold is chosen from out-of-fold predictions (each training grant
# is scored by a model that never saw it), never from the test set.
# ---------------------------------------------------------------------------
def make_classifier():
    # class_weight="balanced" because most sentences are not requirements
    return LogisticRegression(class_weight="balanced", max_iter=2000) #number of iterations of training (max)


def out_of_fold_probs(X, y, groups): # basically leaves out some amount of grant docs on a training run so it doesnt overfit itself when testing
    """Score every training sentence with a model trained on the OTHER grants."""
    n_splits = min(5, len(set(groups)))
    if n_splits < 2:
        raise ValueError("Need at least 2 training grants.")
    probs = np.full(len(y), np.nan)
    for fit_idx, score_idx in GroupKFold(n_splits=n_splits).split(X, y, groups):
        if len(set(y[fit_idx])) < 2:  # this fold has no positives to learn from
            continue
        clf = make_classifier().fit(X[fit_idx], y[fit_idx])
        probs[score_idx] = clf.predict_proba(X[score_idx])[:, 1]
    return probs


def pick_threshold(y, probs, target_recall):
#sort of like when you curve a grade, this helps determine what sentences get passed by calculating a relative "pass" threshold
# target recall = how much recall you want to maintain, e.g. higher thresholds can lower the recall so don't go too aggressive
    """Highest threshold that still catches target_recall of the positives."""
    pos = np.sort(probs[(y == 1) & ~np.isnan(probs)])
    k = int(np.floor((1 - target_recall) * len(pos)))
    return float(pos[k])


first_pass = {}
for col in LABEL_COLUMNS:
    y = train_df[col].to_numpy()
    n_pos = int(y.sum())
    if n_pos < MIN_POSITIVES:
        print(f"{col}: only {n_pos} positives, skipped (stays zero-shot)")
        continue
    oof = out_of_fold_probs(X_train, y, train_groups)
    threshold = pick_threshold(y, oof, TARGET_RECALL)
    clf = make_classifier().fit(X_train, y)
    first_pass[col] = {"clf": clf, "threshold": threshold}
    print(f"{col}: trained on {n_pos} positives, threshold {threshold:.3f}")

In [ ]:
# ---------------------------------------------------------------------------
# CELL 6: test-set results, trained vs. zero-shot.
# Recall matters most: a requirement missed here never reaches the NLI step.
# ---------------------------------------------------------------------------
def recall_precision(y, pred):
    return (recall_score(y, pred, zero_division=0),
            precision_score(y, pred, zero_division=0))


for col in LABEL_COLUMNS:
    y = test_df[col].to_numpy()
    print("\n" + "=" * 70)
    print(col)
    if y.sum() == 0:
        print("  No positive sentences in the test grants, nothing to measure.")
        continue

    zs_pred = zero_shot_scores(embedder, X_test, QUERIES[col]) >= ZERO_SHOT_MIN_SIM
    r, p = recall_precision(y, zs_pred)
    print(f"  zero-shot  recall {r:.2f}  precision {p:.2f}")

    if col not in first_pass:
        print("  (no trained model for this label)")
        continue
    entry = first_pass[col]
    probs = entry["clf"].predict_proba(X_test)[:, 1]
    pred = probs >= entry["threshold"]
    r, p = recall_precision(y, pred)
    print(f"  trained    recall {r:.2f}  precision {p:.2f}")

    missed = test_df.sentence[(y == 1) & ~pred]
    if len(missed):
        print(f"  Missed by trained model ({len(missed)}):")
        for s in missed.head(5):
            print(f'    - "{s}"')

In [ ]:
# ---------------------------------------------------------------------------
# CELL 7: save. Only save if the trained results beat zero-shot in Cell 6.
# grant_eligibility uses a trained classifier for each label saved here
# and zero-shot for the rest.
# ---------------------------------------------------------------------------
MODEL_OUT.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "embed_model": EMBED_MODEL,
        "labels": first_pass,
        "train_grants": sorted(set(train_groups)),
        "test_grants": TEST_GRANTS,
        "target_recall": TARGET_RECALL,
    },
    MODEL_OUT,
)
print(f"Saved {sorted(first_pass)} to {MODEL_OUT}")